# Implementasi Morphological Filtering untuk Mengurangi Noise pada Citra Tulisan Menggunakan Operasi Morfologi Matematis

**Mata kuliah:** Pengolahan Citra Digital

**Cara pakai:** jalankan cell dari atas ke bawah dengan `Shift + Enter`. Cell 2 akan meminta kamu mengunggah satu gambar tulisan.

**Alur:** Input -> Preprocessing -> Pembuatan noise -> Morphological filtering -> Evaluasi -> Output

## Cell 1 - Import library dan konfigurasi
Menyiapkan library, folder `hasil/`, random seed (agar noise bisa direproduksi), dan fungsi `tampilkan()` dengan skala hitam-putih yang konsisten.

In [ ]:
import os
import shutil
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Folder untuk menyimpan semua output
FOLDER_HASIL = "hasil"
os.makedirs(FOLDER_HASIL, exist_ok=True)

# Random seed agar noise bisa direproduksi (hasil sama setiap dijalankan)
SEED = 42

# Pengaturan tampilan grafik
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["axes.titlesize"] = 11

def tampilkan(ax, citra, judul):
    """Menampilkan citra pada sumbu matplotlib dengan skala warna konsisten."""
    if citra.ndim == 3:  # citra berwarna (BGR -> RGB)
        ax.imshow(cv2.cvtColor(citra, cv2.COLOR_BGR2RGB))
    else:                # citra grayscale/biner, skala tetap 0-255
        ax.imshow(citra, cmap="gray", vmin=0, vmax=255)
    ax.set_title(judul)
    ax.axis("off")

print("Versi OpenCV :", cv2.__version__)
print("Versi NumPy  :", np.__version__)
print("Folder hasil :", os.path.abspath(FOLDER_HASIL))

**Hasil yang seharusnya muncul:** versi OpenCV, versi NumPy, dan lokasi folder `hasil`.

## Cell 2 - Upload citra tulisan
Klik **Choose Files** dan pilih satu gambar tulisan (jpg/png). Program menerima gambar berwarna maupun grayscale. Satu gambar ini dipakai untuk **seluruh** eksperimen agar perbandingan adil.

Jika belum punya gambar, ubah `GUNAKAN_CONTOH = True` untuk memakai gambar tulisan buatan program (hanya untuk mencoba).

In [ ]:
MAKS_LEBAR = 800          # citra lebih lebar dari ini akan diperkecil
GUNAKAN_CONTOH = False    # ubah ke True jika belum punya gambar tulisan

if GUNAKAN_CONTOH:
    citra_asli = np.full((300, 800, 3), 255, np.uint8)
    cv2.putText(citra_asli, "HALO ABC", (40, 200), cv2.FONT_HERSHEY_SIMPLEX,
                4, (0, 0, 0), 12, cv2.LINE_AA)
    nama_file = "contoh_sintetis"
else:
    from google.colab import files
    print("Silakan pilih SATU file gambar tulisan (jpg/png).")
    upload = files.upload()
    if len(upload) == 0:
        raise RuntimeError("Tidak ada file yang diunggah. Jalankan ulang cell ini.")
    nama_file = list(upload.keys())[0]
    citra_asli = cv2.imread(nama_file, cv2.IMREAD_UNCHANGED)
    if citra_asli is None:
        raise RuntimeError(f"File '{nama_file}' tidak bisa dibaca sebagai gambar. Gunakan format jpg/png.")

# Jika PNG punya transparansi (alpha), tempelkan di atas latar putih
if citra_asli.ndim == 3 and citra_asli.shape[2] == 4:
    alpha = citra_asli[:, :, 3:4].astype(np.float32) / 255.0
    rgb = citra_asli[:, :, :3].astype(np.float32)
    citra_asli = (rgb * alpha + 255 * (1 - alpha)).astype(np.uint8)

# Jika bukan uint8 (misalnya 16-bit), normalisasi ke 0-255
if citra_asli.dtype != np.uint8:
    citra_asli = cv2.normalize(citra_asli, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

# Perkecil jika terlalu lebar
tinggi, lebar = citra_asli.shape[:2]
if lebar > MAKS_LEBAR:
    skala = MAKS_LEBAR / lebar
    citra_asli = cv2.resize(citra_asli, (MAKS_LEBAR, int(tinggi * skala)),
                            interpolation=cv2.INTER_AREA)

print("File       :", nama_file)
print("Dimensi    :", citra_asli.shape)
print("Tipe data  :", citra_asli.dtype)

fig, ax = plt.subplots(figsize=(10, 5))
tampilkan(ax, citra_asli, "Citra asli (input)")
plt.show()

**Hasil yang seharusnya muncul:** nama file, dimensi, tipe data `uint8`, dan tampilan citra asli.

## Cell 3 - Preprocessing
Grayscale -> Otsu thresholding -> pastikan tulisan putih (255) dan latar hitam (0). Hasil disimpan sebagai **ground truth** dan dikunci agar tidak bisa tertimpa.

Hasil thresholding **harus diperiksa secara visual**, karena pencahayaan, bayangan, atau warna latar dapat memengaruhi pemisahan tulisan dan latar.

In [ ]:
# 1. Konversi ke grayscale (jika masih berwarna)
if citra_asli.ndim == 3:
    gray = cv2.cvtColor(citra_asli, cv2.COLOR_BGR2GRAY)
else:
    gray = citra_asli.copy()

# 2. Blur ringan untuk menstabilkan Otsu, lalu thresholding
blur = cv2.GaussianBlur(gray, (3, 3), 0)
_, biner = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# 3. Pastikan TULISAN = putih (255), LATAR = hitam (0)
#    Asumsi: tulisan adalah bagian yang lebih sedikit dibanding latar.
#    Jika hasil otomatis salah, isi PAKSA_INVERSI = True / False secara manual.
PAKSA_INVERSI = None
rasio_putih = np.mean(biner == 255)
perlu_inversi = (rasio_putih > 0.5) if PAKSA_INVERSI is None else PAKSA_INVERSI
if perlu_inversi:
    biner = cv2.bitwise_not(biner)

print(f"Rasio piksel putih sebelum pengecekan inversi: {rasio_putih:.3f}")
print("Inversi diterapkan:", perlu_inversi)
print("Rasio piksel putih (tulisan) sekarang:", round(float(np.mean(biner == 255)), 3))

# 4. Tampilkan
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
tampilkan(axes[0], citra_asli, "Citra asli")
tampilkan(axes[1], gray, "Grayscale")
tampilkan(axes[2], biner, "Citra biner (tulisan putih)")
plt.tight_layout()
plt.show()

# 5. Simpan sebagai GROUND TRUTH dan kunci agar tidak bisa ditimpa
ground_truth = biner.astype(np.uint8).copy()
assert set(np.unique(ground_truth)).issubset({0, 255}), "Citra biner harus hanya berisi 0 dan 255"
ground_truth.setflags(write=False)   # array dikunci: upaya mengubahnya akan error
cv2.imwrite(f"{FOLDER_HASIL}/ground_truth.png", ground_truth)

print("Ground truth tersimpan. Dimensi:", ground_truth.shape, "| dtype:", ground_truth.dtype)

**Hasil yang seharusnya muncul:** tiga gambar (asli, grayscale, biner). Pada citra biner, huruf harus **putih di atas latar hitam** dan terlihat utuh. Jika terbalik atau huruf bolong, atur `PAKSA_INVERSI` (True/False) atau ganti gambar.

## Cell 4 - Membuat variasi noise sintetis
Empat kondisi eksperimen dibuat dari ground truth yang **sama** (selalu memakai `.copy()`, jadi ground truth tidak berubah):
1. **Salt-and-pepper**: titik putih acak di latar dan titik hitam acak di goresan.
2. **Bintik putih**: titik putih kecil hanya di latar yang jauh dari huruf.
3. **Lubang hitam**: lubang kecil di dalam goresan.
4. **Goresan terputus**: celah kecil (2-3 piksel) memotong goresan.

Ubah parameter `p`, `jumlah`, `radius_maks` untuk membuat tingkat noise rendah/tinggi.

In [ ]:
def validasi_citra(citra, acuan, nama):
    """Memastikan citra uint8, ukuran sama dengan acuan, dan hanya berisi 0/255."""
    assert citra.dtype == np.uint8, f"{nama}: dtype harus uint8"
    assert citra.shape == acuan.shape, f"{nama}: ukuran berbeda dari ground truth"
    assert set(np.unique(citra)).issubset({0, 255}), f"{nama}: nilai harus 0 atau 255"

# ---------- Eksperimen 1: Salt-and-pepper ----------
def noise_salt_pepper(gt, p=0.03, seed=SEED):
    """p = probabilitas total noise. Separuh jadi 'salt' (putih), separuh 'pepper' (hitam).
    Salt hanya terlihat di latar hitam; pepper hanya terlihat di goresan putih."""
    rng = np.random.default_rng(seed)
    hasil = gt.copy()
    acak = rng.random(gt.shape)
    hasil[acak < p / 2] = 255        # salt: titik putih
    hasil[acak > 1 - p / 2] = 0      # pepper: titik hitam
    return hasil

# ---------- Eksperimen 2: Bintik putih di latar ----------
def noise_bintik_putih(gt, jumlah=150, radius_maks=2, seed=SEED + 1):
    """Bintik putih kecil (radius 0..radius_maks) hanya di latar yang jauh dari huruf,
    sehingga benar-benar terpisah dari goresan."""
    rng = np.random.default_rng(seed)
    hasil = gt.copy()
    jarak = radius_maks + 2
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * jarak + 1, 2 * jarak + 1))
    zona_aman = cv2.dilate(gt, k) == 0          # latar yang jauh dari huruf
    ys, xs = np.nonzero(zona_aman)
    if len(xs) == 0:
        raise RuntimeError("Tidak ada area latar yang cukup luas untuk bintik.")
    idx = rng.choice(len(xs), size=min(jumlah, len(xs)), replace=False)
    for i in idx:
        r = int(rng.integers(0, radius_maks + 1))
        cv2.circle(hasil, (int(xs[i]), int(ys[i])), r, 255, -1)
    return hasil

# ---------- Eksperimen 3: Lubang hitam pada goresan ----------
def noise_lubang_hitam(gt, jumlah=200, radius_maks=2, seed=SEED + 2):
    """Lubang hitam kecil di bagian dalam goresan putih."""
    rng = np.random.default_rng(seed)
    hasil = gt.copy()
    bagian_dalam = cv2.erode(gt, np.ones((3, 3), np.uint8))   # piksel bukan di tepi
    ys, xs = np.nonzero(bagian_dalam)
    if len(xs) == 0:                                          # cadangan jika goresan sangat tipis
        ys, xs = np.nonzero(gt)
    idx = rng.choice(len(xs), size=min(jumlah, len(xs)), replace=False)
    for i in idx:
        r = int(rng.integers(0, radius_maks + 1))
        cv2.circle(hasil, (int(xs[i]), int(ys[i])), r, 0, -1)
    return hasil

# ---------- Eksperimen 4: Goresan terputus ----------
def arah_goresan(gt, x, y, r=7):
    """Memperkirakan arah goresan di sekitar (x,y) dengan PCA sederhana."""
    h, w = gt.shape
    y0, y1 = max(0, y - r), min(h, y + r + 1)
    x0, x1 = max(0, x - r), min(w, x + r + 1)
    ys, xs = np.nonzero(gt[y0:y1, x0:x1])
    if len(xs) < 3:
        return None
    koord = np.stack([xs, ys], axis=1).astype(np.float64)
    koord -= koord.mean(axis=0)
    _, vektor = np.linalg.eigh(np.cov(koord.T))
    return vektor[:, -1]   # arah utama (sumbu terpanjang) goresan

def noise_goresan_terputus(gt, jumlah=30, lebar_maks=3, seed=SEED + 3):
    """Memotong goresan dengan garis hitam pendek TEGAK LURUS arah goresan.
    Lebar celah acak 2..lebar_maks piksel (dijaga kecil agar masih bisa diuji)."""
    rng = np.random.default_rng(seed)
    hasil = gt.copy()
    dist = cv2.distanceTransform(gt, cv2.DIST_L2, 5)
    lokal_maks = cv2.dilate(dist, np.ones((5, 5), np.uint8))
    punggung = (dist >= lokal_maks) & (dist >= 1.0)   # titik di tengah goresan
    ys, xs = np.nonzero(punggung)
    if len(xs) == 0:
        raise RuntimeError("Goresan tidak ditemukan pada citra biner.")
    idx = rng.choice(len(xs), size=min(jumlah, len(xs)), replace=False)
    for i in idx:
        x, y = int(xs[i]), int(ys[i])
        arah = arah_goresan(gt, x, y)
        if arah is None:
            continue
        tegak = np.array([-arah[1], arah[0]])          # tegak lurus arah goresan
        setengah = dist[y, x] + 3                      # cukup panjang untuk memotong
        p1 = (int(round(x - tegak[0] * setengah)), int(round(y - tegak[1] * setengah)))
        p2 = (int(round(x + tegak[0] * setengah)), int(round(y + tegak[1] * setengah)))
        lebar = int(rng.integers(2, lebar_maks + 1))
        cv2.line(hasil, p1, p2, 0, thickness=lebar)
    return hasil

# ---------- Buat semua citra noisy dari ground truth yang SAMA ----------
noisy = {
    "salt_pepper":      noise_salt_pepper(ground_truth, p=0.03),
    "bintik_putih":     noise_bintik_putih(ground_truth, jumlah=150),
    "lubang_hitam":     noise_lubang_hitam(ground_truth, jumlah=200),
    "goresan_terputus": noise_goresan_terputus(ground_truth, jumlah=30),
}

for nama, citra in noisy.items():
    validasi_citra(citra, ground_truth, nama)
    cv2.imwrite(f"{FOLDER_HASIL}/noisy_{nama}.png", citra)

# Pemeriksaan: bintik putih tidak boleh menyentuh area huruf
assert np.all(noisy["bintik_putih"][ground_truth == 255] == 255)
print("Ground truth masih read-only:", not ground_truth.flags.writeable)

# ---------- Tampilkan ----------
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
daftar = [("Ground truth", ground_truth)] + [(f"Noisy: {n}", c) for n, c in noisy.items()]
for ax, (judul, citra) in zip(axes.ravel(), daftar):
    tampilkan(ax, citra, judul)
axes.ravel()[-1].axis("off")
plt.tight_layout()
plt.show()

**Hasil yang seharusnya muncul:** ground truth dan empat citra noisy. Zoom gambar untuk memastikan bintik, lubang, dan celah terlihat jelas tetapi masih kecil.

## Cell 5 - Implementasi morphological filtering
Satu fungsi `terapkan_morfologi()` dipakai untuk semua noise. **Keempat** operasi (erosion, dilation, opening, closing) dicoba pada **setiap** citra noisy dengan kernel 3x3 dan 5x5, bentuk persegi dan elips, sehingga perbandingannya adil.

Opening diharapkan efektif untuk bintik putih terpisah dan closing untuk lubang/celah kecil, tetapi itu masih **hipotesis** yang dibuktikan di Cell 7. Efektivitas bergantung pada ukuran dan bentuk noise, ketebalan huruf, serta ukuran dan bentuk kernel.

In [ ]:
OPERASI = ["erosion", "dilation", "opening", "closing"]
UKURAN  = [3, 5]
BENTUK  = ["persegi", "elips"]

def buat_kernel(ukuran, bentuk):
    """Membuat structuring element persegi atau elips."""
    jenis = {"persegi": cv2.MORPH_RECT, "elips": cv2.MORPH_ELLIPSE}
    if bentuk not in jenis:
        raise ValueError("Bentuk kernel harus 'persegi' atau 'elips'.")
    return cv2.getStructuringElement(jenis[bentuk], (ukuran, ukuran))

def terapkan_morfologi(citra, operasi, ukuran=3, bentuk="persegi"):
    """Menerapkan satu operasi morfologi pada citra biner (foreground putih)."""
    assert citra.dtype == np.uint8, "Citra harus bertipe uint8"
    kernel = buat_kernel(ukuran, bentuk)
    if operasi == "erosion":
        return cv2.erode(citra, kernel)                         # menipiskan foreground
    elif operasi == "dilation":
        return cv2.dilate(citra, kernel)                        # menebalkan foreground
    elif operasi == "opening":
        return cv2.morphologyEx(citra, cv2.MORPH_OPEN, kernel)  # erosion -> dilation
    elif operasi == "closing":
        return cv2.morphologyEx(citra, cv2.MORPH_CLOSE, kernel) # dilation -> erosion
    else:
        raise ValueError(f"Operasi tidak dikenal: {operasi}")

# Contoh bentuk kernel (untuk dipahami)
print("Kernel persegi 3x3:\n", buat_kernel(3, "persegi"))
print("\nKernel elips 5x5:\n", buat_kernel(5, "elips"))

# Terapkan SEMUA operasi pada SETIAP citra noisy (citra noisy tidak diubah)
hasil_filter = {}
for nama, citra_noisy in noisy.items():
    for op in OPERASI:
        for u in UKURAN:
            for b in BENTUK:
                hasil_filter[(nama, op, u, b)] = terapkan_morfologi(citra_noisy, op, u, b)

print(f"\nTotal citra hasil filtering: {len(hasil_filter)}")

**Hasil yang seharusnya muncul:** matriks kernel dan total 64 citra hasil (4 noise x 4 operasi x 2 ukuran x 2 bentuk). Perhatikan kernel elips 3x3 di OpenCV berbentuk tanda plus, sehingga berbeda dari persegi 3x3.

## Cell 6 - Visualisasi perbandingan
Untuk tiap jenis noise dan ukuran kernel dibuat gambar 2x3: ground truth, noisy, erosion, dilation, opening, closing. Kernel persegi ditampilkan; kernel elips disimpan sebagai PNG di folder `hasil/`.

In [ ]:
def tampilkan_perbandingan(nama_noise, ukuran, bentuk, tampil=True):
    """Menampilkan: ground truth, noisy, erosion, dilation, opening, closing."""
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    gambar = [("Ground truth (biner asli)", ground_truth),
              (f"Noisy: {nama_noise}", noisy[nama_noise])]
    for op in OPERASI:
        gambar.append((op.capitalize(), hasil_filter[(nama_noise, op, ukuran, bentuk)]))

    for ax, (judul, citra) in zip(axes.ravel(), gambar):
        tampilkan(ax, citra, judul)

    fig.suptitle(f"{nama_noise} | kernel {ukuran}x{ukuran} {bentuk}", fontsize=14)
    plt.tight_layout()
    fig.savefig(f"{FOLDER_HASIL}/perbandingan_{nama_noise}_{ukuran}x{ukuran}_{bentuk}.png",
                dpi=150, bbox_inches="tight")
    if tampil:
        plt.show()
    else:
        plt.close(fig)

# Tampilkan kernel persegi 3x3 dan 5x5 untuk tiap jenis noise
for nama in noisy:
    for u in UKURAN:
        tampilkan_perbandingan(nama, u, "persegi", tampil=True)

# Kernel elips: hanya disimpan sebagai PNG (ubah tampil=True jika ingin melihatnya)
for nama in noisy:
    for u in UKURAN:
        tampilkan_perbandingan(nama, u, "elips", tampil=False)

**Hasil yang seharusnya muncul:** 8 gambar perbandingan. Cara membaca: bandingkan setiap hasil dengan *Ground truth*; lihat apakah bintik hilang, lubang tertutup, celah tersambung, dan apakah huruf menjadi terlalu tipis atau tebal.

## Cell 7 - Evaluasi hasil
Metrik dihitung terhadap **ground truth bersih**, bukan citra noisy. Baris "tanpa filter (noisy)" dimasukkan sebagai pembanding: filtering baru berguna jika IoU-nya lebih tinggi dari baris ini.

- IoU dan F1 lebih tinggi = piksel foreground lebih sesuai dengan ground truth.
- Jumlah piksel putih atau connected components saja tidak cukup untuk menentukan metode terbaik.
- Metrik harus dilengkapi pemeriksaan visual, karena huruf yang rusak masih bisa menghasilkan nilai yang tampak cukup baik.

In [ ]:
def hitung_metrik(gt, hasil):
    """IoU dan F1-score dengan piksel putih (255) sebagai foreground,
    dibandingkan terhadap ground truth yang BERSIH (bukan citra noisy)."""
    assert gt.shape == hasil.shape, "Ukuran citra harus sama"
    gt_b, h_b = (gt == 255), (hasil == 255)
    tp = np.sum(gt_b & h_b)        # putih benar
    fp = np.sum(~gt_b & h_b)       # putih tapi seharusnya hitam
    fn = np.sum(gt_b & ~h_b)       # hitam tapi seharusnya putih
    iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0
    f1  = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0.0
    return float(iou), float(f1)

def hitung_komponen(citra):
    """Jumlah connected components foreground (tidak termasuk latar)."""
    n_label, _ = cv2.connectedComponents(citra, connectivity=8)
    return int(n_label - 1)

print("Ground truth -> piksel foreground:", int(np.sum(ground_truth == 255)),
      "| connected components:", hitung_komponen(ground_truth))

baris = []

# Baris pembanding: citra noisy TANPA filtering
for nama, c in noisy.items():
    iou, f1 = hitung_metrik(ground_truth, c)
    baris.append({"Jenis noise": nama, "Operasi morfologi": "tanpa filter (noisy)",
                  "Ukuran kernel": "-", "Bentuk kernel": "-",
                  "Jumlah piksel foreground": int(np.sum(c == 255)),
                  "Jumlah connected components": hitung_komponen(c),
                  "IoU": iou, "F1-score": f1})

# Semua hasil filtering
for (nama, op, u, b), h in hasil_filter.items():
    iou, f1 = hitung_metrik(ground_truth, h)
    baris.append({"Jenis noise": nama, "Operasi morfologi": op,
                  "Ukuran kernel": f"{u}x{u}", "Bentuk kernel": b,
                  "Jumlah piksel foreground": int(np.sum(h == 255)),
                  "Jumlah connected components": hitung_komponen(h),
                  "IoU": iou, "F1-score": f1})

df = pd.DataFrame(baris)
df["IoU"] = df["IoU"].round(4)
df["F1-score"] = df["F1-score"].round(4)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)

print("\n=== Seluruh hasil, diurutkan berdasarkan IoU (tertinggi di atas) ===")
df_urut = df.sort_values(["IoU", "F1-score"], ascending=False).reset_index(drop=True)
display(df_urut)

print("\n=== 5 konfigurasi teratas untuk tiap jenis noise ===")
for nama in noisy:
    print(f"\n>> {nama}")
    display(df[df["Jenis noise"] == nama]
            .sort_values("IoU", ascending=False).head(5).reset_index(drop=True))

**Hasil yang seharusnya muncul:** tabel lengkap (68 baris: 4 baris pembanding + 64 hasil filtering) dan 5 konfigurasi teratas per jenis noise. Jangan menyimpulkan metode terbaik sebelum tabel ini muncul.

## Cell 8 - Penyimpanan hasil dan unduh ZIP
Semua citra, tabel CSV, dan gambar perbandingan disimpan di `hasil/` dengan nama yang memuat jenis noise, operasi, dan ukuran kernel (contoh: `filter_bintik_putih_opening_3x3_persegi.png`). Lalu dibuat `hasil_morfologi.zip`.

In [ ]:
# 1. Ground truth & citra noisy
cv2.imwrite(f"{FOLDER_HASIL}/ground_truth.png", ground_truth)
for nama, c in noisy.items():
    cv2.imwrite(f"{FOLDER_HASIL}/noisy_{nama}.png", c)

# 2. Semua citra hasil filtering. Nama file memuat: noise, operasi, ukuran kernel, bentuk kernel
for (nama, op, u, b), h in hasil_filter.items():
    cv2.imwrite(f"{FOLDER_HASIL}/filter_{nama}_{op}_{u}x{u}_{b}.png", h)

# 3. Tabel evaluasi
df_urut.to_csv(f"{FOLDER_HASIL}/evaluasi_morfologi.csv", index=False)

# 4. Kumpulkan dalam ZIP lalu unduh
def unduh_zip():
    shutil.make_archive("hasil_morfologi", "zip", FOLDER_HASIL)
    print("ZIP dibuat: hasil_morfologi.zip")
    try:
        from google.colab import files
        files.download("hasil_morfologi.zip")
    except Exception as e:
        print("Unduhan otomatis gagal:", e,
              "\nBuka panel Files (ikon folder di kiri) lalu unduh manual.")

print("Jumlah file di folder hasil:", len(os.listdir(FOLDER_HASIL)))
unduh_zip()

**Hasil yang seharusnya muncul:** jumlah file di folder `hasil`, lalu browser mengunduh `hasil_morfologi.zip`. Jika diblokir browser, ambil dari panel **Files** di kiri Colab.

## Cell 9 - Analisis otomatis dan kesimpulan
Cell ini menyusun analisis dari tabel `df` (tanpa kesimpulan yang ditulis sebelumnya): konfigurasi terbaik per noise, perbandingan 3x3 vs 5x5, kasus "lebih bersih tapi IoU turun", keterbatasan, dan draft kesimpulan. Draft ini adalah **bahan laporan**; wajib dilengkapi pemeriksaan visual.

In [ ]:
df_filter = df[df["Operasi morfologi"] != "tanpa filter (noisy)"].copy()
df_dasar  = df[df["Operasi morfologi"] == "tanpa filter (noisy)"].set_index("Jenis noise")
komp_gt   = hitung_komponen(ground_truth)

# ---------- 1. Konfigurasi terbaik per jenis noise ----------
print("=" * 70)
print("1. KONFIGURASI DENGAN IoU / F1-SCORE TERTINGGI PER JENIS NOISE")
print("=" * 70)
idx_terbaik = df_filter.groupby("Jenis noise")["IoU"].idxmax()
terbaik = df_filter.loc[idx_terbaik].set_index("Jenis noise")
terbaik["IoU noisy (tanpa filter)"] = df_dasar["IoU"]
terbaik["Selisih IoU"] = (terbaik["IoU"] - terbaik["IoU noisy (tanpa filter)"]).round(4)
display(terbaik[["Operasi morfologi", "Ukuran kernel", "Bentuk kernel",
                 "IoU", "F1-score", "IoU noisy (tanpa filter)", "Selisih IoU"]])

# ---------- 2. Perbandingan kernel 3x3 vs 5x5 ----------
print("\n" + "=" * 70)
print("2. PERBANDINGAN KERNEL 3x3 vs 5x5 (IoU tertinggi di antara bentuk kernel)")
print("=" * 70)
pivot = df_filter.pivot_table(index=["Jenis noise", "Operasi morfologi"],
                              columns="Ukuran kernel", values="IoU", aggfunc="max")
pivot["Selisih (5x5 - 3x3)"] = (pivot["5x5"] - pivot["3x3"]).round(4)
pivot["Lebih baik"] = np.where(pivot["Selisih (5x5 - 3x3)"] > 0, "5x5",
                      np.where(pivot["Selisih (5x5 - 3x3)"] < 0, "3x3", "sama"))
display(pivot)

# ---------- 3. Noise berkurang, tetapi bentuk huruf rusak ----------
print("\n" + "=" * 70)
print("3. KASUS 'LEBIH BERSIH' TETAPI KESESUAIAN DENGAN GROUND TRUTH TURUN")
print("   (jumlah komponen lebih dekat ke ground truth, namun IoU lebih rendah dari citra noisy)")
print("=" * 70)
kasus = df_filter.copy()
kasus["IoU noisy"] = kasus["Jenis noise"].map(df_dasar["IoU"])
kasus["Komponen noisy"] = kasus["Jenis noise"].map(df_dasar["Jumlah connected components"])
lebih_bersih = (kasus["Jumlah connected components"] - komp_gt).abs() < (kasus["Komponen noisy"] - komp_gt).abs()
iou_turun = kasus["IoU"] < kasus["IoU noisy"]
rusak = kasus[lebih_bersih & iou_turun]
print(f"Jumlah konfigurasi yang masuk kategori ini: {len(rusak)} dari {len(kasus)}")
display(rusak[["Jenis noise", "Operasi morfologi", "Ukuran kernel", "Bentuk kernel",
               "Jumlah connected components", "IoU", "IoU noisy"]]
        .sort_values("IoU").head(10))
print(f"Total konfigurasi yang justru menurunkan IoU dibanding noisy: {int(iou_turun.sum())}")
print("-> Buka gambar perbandingan_*.png untuk memeriksa apakah huruf menipis, hilang, atau menyatu.")

# ---------- 4. Keterbatasan (angka + panduan pemeriksaan visual) ----------
print("\n" + "=" * 70)
print("4. KETERBATASAN: sisa selisih dari ground truth pada konfigurasi terbaik")
print("=" * 70)
for nama, r in terbaik.iterrows():
    print(f"- {nama}: IoU terbaik = {r['IoU']:.4f} -> masih ada selisih {1 - r['IoU']:.4f} dari citra asli.")
print("""
Periksa secara visual dan jawab bersama kelompok:
  a. Pada noise mana IoU terbaik masih rendah? Apakah noise menyerupai goresan huruf?
  b. Apakah kernel 5x5 menghapus bagian huruf yang tipis atau menyatukan huruf berdekatan?
  c. Apakah celah yang lebih lebar dari kernel tetap tidak tersambung oleh closing?
""")

# ---------- 5. Draft kesimpulan dari hasil aktual ----------
print("=" * 70)
print("5. DRAFT KESIMPULAN (disusun otomatis dari angka; edit setelah memeriksa gambar)")
print("=" * 70)
kalimat = []
for nama, r in terbaik.iterrows():
    iou_n = r["IoU noisy (tanpa filter)"]
    if r["Selisih IoU"] > 0:
        kalimat.append(
            f"Pada noise '{nama}', konfigurasi terbaik adalah {r['Operasi morfologi']} "
            f"kernel {r['Ukuran kernel']} bentuk {r['Bentuk kernel']} dengan IoU {r['IoU']:.4f} "
            f"dan F1-score {r['F1-score']:.4f}, naik dari IoU citra noisy {iou_n:.4f} "
            f"(peningkatan {r['Selisih IoU']:.4f}).")
    else:
        kalimat.append(
            f"Pada noise '{nama}', tidak ada konfigurasi yang melampaui IoU citra noisy "
            f"({iou_n:.4f}); filtering tidak memperbaiki kesesuaian piksel pada kondisi ini.")
kalimat.append(
    f"Sebanyak {int(iou_turun.sum())} dari {len(kasus)} konfigurasi justru menurunkan IoU "
    f"dibanding citra noisy, sehingga pemilihan operasi harus disesuaikan dengan jenis noise.")
kalimat.append(
    "Kesimpulan akhir harus dikonfirmasi dengan pemeriksaan visual terhadap gambar perbandingan.")

teks_kesimpulan = "\n".join(f"{i}. {k}" for i, k in enumerate(kalimat, 1))
print(teks_kesimpulan)

with open(f"{FOLDER_HASIL}/analisis_kesimpulan.txt", "w", encoding="utf-8") as f:
    f.write(teks_kesimpulan)

# Perbarui ZIP agar menyertakan file analisis
unduh_zip()

**Hasil yang seharusnya muncul:** lima bagian analisis dan draft kesimpulan. ZIP diunduh ulang dan kini berisi `analisis_kesimpulan.txt`.

---
*Selesai. Salin angka dari tabel dan gambar dari folder `hasil/` ke laporan, lalu tulis interpretasi visual sendiri.*